# Zero shot notebook

IDEAS:

- Coger 2 modelos, por ejemplo, Mistral 7B instruc y Qwen del otro notebook

- Probar zero-shot directamente

- Hacerles fine-tuning

- Probar de nuevo zeroshot y ver si hay mejora

La idea en este notebook es probar con una estrategia de zero shot, posiblemente en distintos modelos.

También se puede probar son un prompt más elaborado y otro más sencillito.

In [1]:
import json
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
!pip install unsloth
from unsloth import FastLanguageModel
import gc

/tmp/ipykernel_797/2560775310.py:6: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SYSTEM_PROMPT = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

SYSTEM_PROMPT_2 = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Debes responder EXCLUSIVAMENTE con un objeto JSON válido, sin incluir explicaciones previas ni posteriores.
El formato debe ser ESTRICTAMENTE este:
{
  "explicacion": "Aquí escribes una breve explicación de la respuesta elegida basándote en el texto.",
  "respuesta": "Aquí escribes SOLAMENTE la letra de la opción correcta (A, B, C, D...)."
}"""

## Modelo [Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen3.5-9B)

In [5]:

max_seq_length = 4096
dtype = None
load_in_4bit = True

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen

### zero-shot

In [ ]:
import json
import torch
import re
from tqdm import tqdm

# 1. Cargar ambos ficheros
with open('multiple_choice.json', 'r', encoding='utf-8') as f:
    main_data = json.load(f)

with open('subset_100.json', 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

def evaluar_modelo_subset():
    aciertos = 0
    total_procesadas = 0
    stats_por_nivel = {} # Para ver dónde falla más el modelo

    print(f"--- Iniciando evaluación sobre {len(ground_truth)} preguntas ---")

    for exam in main_data['exams']:
        nivel = exam['level']
        if nivel not in stats_por_nivel:
            stats_por_nivel[nivel] = {"aciertos": 0, "total": 0}

        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']
            texto_contexto = exercise.get('text', '')

            for question in exercise['questions']:
                q_id = question['questionId']

                # Solo procesamos si la pregunta está en nuestro subset de 100
                if q_id in ground_truth:
                    respuesta_correcta = ground_truth[q_id]

                    # Preparar opciones y Prompt
                    opciones_str = "\n".join([f"{o['optionId']}) {o['text']}" for o in question['options']])

                    messages = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": f"Texto:\n{texto_contexto}\n\nPregunta: {question['text']}\nOpciones:\n{opciones_str}\n\nRespuesta:"}
                    ]

                    # Inferencia
                    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

                    with torch.no_grad():
                        outputs = model.generate(
                            input_ids = inputs,
                            max_new_tokens = 5,
                            do_sample = False,
                            pad_token_id = tokenizer.eos_token_id
                        )

                    # Decodificar y limpiar
                    gen_tokens = outputs[0][len(inputs[0]):]
                    res_bruta = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip().upper()
                    match = re.search(r'[A-D]', res_bruta)
                    prediccion = match.group(0) if match else "N/A"

                    # Comparar
                    es_correcto = (prediccion == respuesta_correcta)
                    if es_correcto:
                        aciertos += 1
                        stats_por_nivel[nivel]["aciertos"] += 1

                    stats_por_nivel[nivel]["total"] += 1
                    total_procesadas += 1

    # --- RESULTADOS FINALES ---
    accuracy_total = (aciertos / total_procesadas) * 100 if total_procesadas > 0 else 0

    print("\n" + "="*40)
    print(f"📊 RESULTADOS GLOBALES")
    print(f"Total preguntas: {total_procesadas}")
    print(f"Aciertos: {aciertos}")
    print(f"Accuracy Total: {accuracy_total:.2f}%")
    print("="*40)

    print("\n📈 DESGLOSE POR NIVEL:")
    for nivel, stats in sorted(stats_por_nivel.items()):
        if stats['total'] > 0:
            acc_nivel = (stats['aciertos'] / stats['total']) * 100
            print(f"Nivel {nivel}: {acc_nivel:.2f}% ({stats['aciertos']}/{stats['total']})")

# Ejecutar la evaluación
evaluar_modelo_subset()

--- Iniciando evaluación sobre 100 preguntas ---

📊 RESULTADOS GLOBALES
Total preguntas: 100
Aciertos: 81
Accuracy Total: 81.00%

📈 DESGLOSE POR NIVEL:
Nivel A1: 83.33% (30/36)
Nivel A2: 78.26% (18/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 75.00% (15/20)


### zero-shot con explicación

In [12]:
import json
import torch
import re
from tqdm import tqdm

path="/content/drive/MyDrive/resultados_qwen.json"

# --- 1. CONFIGURACIÓN DEL TOKENIZER PARA PROCESAMIENTO POR LOTES ---
# El padding a la izquierda es obligatorio para la generación de texto en lotes
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def ejecutar_batch_con_guardado(batch_size=4, output_file='resultados_detallados.json'):
    # Carga de datos
    with open('multiple_choice.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open('subset_100.json', 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)

    # Preparar la lista de tareas (solo las 100 del subset)
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']
            for q in exercise['questions']:
                q_id = q['questionId']
                if q_id in ground_truth:
                    opciones = "\n".join([f"{o['optionId']}) {o['text']}" for o in q['options']])
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": opciones,
                        "real": ground_truth[q_id]
                    })

    resultados_finales = []
    stats = {"total": 0, "aciertos": 0, "por_nivel": {}}

    print(f"🚀 Procesando {len(tareas)} preguntas en lotes de {batch_size}...")

    # Bucle de procesamiento por lotes
    for i in tqdm(range(0, len(tareas), batch_size), desc="Progreso"):
        batch = tareas[i : i + batch_size]

        # Construir mensajes para el lote
        batch_messages = [
            [
                {"role": "system", "content": SYSTEM_PROMPT_2},
                {"role": "user", "content": f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n{t['opciones']}"}
            ] for t in batch
        ]

        # 1. Tokenización y padding (Aseguramos que devuelva tensores y atención)
        model_inputs = tokenizer.apply_chat_template(
            batch_messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            padding=True,
            return_dict=True, # Importante para evitar el error de 'shape'
        ).to("cuda")

        # 2. Inferencia (Pasamos el diccionario desempaquetado)
        with torch.no_grad():
            outputs = model.generate(
                **model_inputs, # Usamos ** para pasar input_ids y attention_mask
                max_new_tokens=400,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # 3. Procesar resultados del lote
        # Usamos model_inputs.input_ids.shape[1] para saber dónde termina el prompt
        input_len = model_inputs.input_ids.shape[1]
        for j, output in enumerate(outputs):
            gen_tokens = output[input_len:]
            res_bruta = tokenizer.decode(gen_tokens, skip_special_tokens=True)

            # Extraer JSON y limpiar respuesta
            prediccion, explicacion = "ERROR", res_bruta
            try:
                json_match = re.search(r'\{.*\}', res_bruta, re.DOTALL)
                if json_match:
                    datos = json.loads(json_match.group(0))
                    letra_raw = datos.get("respuesta", "").strip().upper()
                    match_letra = re.search(r'[A-D]', letra_raw)
                    prediccion = match_letra.group(0) if match_letra else "N/A"
                    explicacion = datos.get("explicacion", "")
            except:
                pass

            # Evaluación y Estadísticas
            real = batch[j]["real"]
            nivel = batch[j]["nivel"]
            es_correcto = (prediccion == real)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1

            # Guardar en la lista para el fichero
            resultados_finales.append({
                "questionId": batch[j]["id"],
                "nivel": nivel,
                "pregunta": batch[j]["pregunta"],
                "respuesta_real": real,
                "prediccion_modelo": prediccion,
                "explicacion": explicacion,
                "estado": "CORRECTO" if es_correcto else "INCORRECTO"
            })

    # --- GUARDADO EN FICHERO ---
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(resultados_finales, f, ensure_ascii=False, indent=4)

    # --- INFORME DE ACCURACY ---
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*45)
    print(f"📊 RESULTADOS GUARDADOS EN: {output_file}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 45)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")
    print("="*45)

# Ejecutar el análisis
ejecutar_batch_con_guardado(batch_size=4, output_file=path)

🚀 Procesando 100 preguntas en lotes de 4...


Progreso: 100%|██████████| 25/25 [09:24<00:00, 22.58s/it]



📊 RESULTADOS GUARDADOS EN: /content/drive/MyDrive/resultados_qwen.json
Accuracy Global: 79.00% (79/100)
---------------------------------------------
Nivel A1: 72.22% (26/36)
Nivel A2: 78.26% (18/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 85.00% (17/20)


### Prueba

In [8]:
import json
import torch
import random
import re

def probar_formato_json(n_pruebas=2):
    # Cargar ficheros
    with open('multiple_choice.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open('subset_100.json', 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)

    # Filtrar preguntas que están en el subset de 100 para poder comparar
    candidatos = []
    for exam in data['exams']:
        for ex_wrapper in exam['exercises']:
            ex = ex_wrapper['exercise']
            for q in ex['questions']:
                if q['questionId'] in ground_truth:
                    candidatos.append({
                        'id': q['questionId'],
                        'nivel': exam['level'],
                        'texto': ex.get('text', ''),
                        'pregunta': q['text'],
                        'opciones': q['options'],
                        'correcta': ground_truth[q['questionId']]
                    })

    # Seleccionar muestras
    muestras = random.sample(candidatos, min(n_pruebas, len(candidatos)))

    print(f"--- TEST DE FORMATO JSON Y EXPLICACIÓN ({len(muestras)} ejemplos) ---\n")

    for i, m in enumerate(muestras):
        opciones_str = "\n".join([f"{o['optionId']}) {o['text']}" for o in m['opciones']])

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_2},
            {"role": "user", "content": f"Texto:\n{m['texto'][:500]}...\n\nPregunta: {m['pregunta']}\nOpciones:\n{opciones_str}"}
        ]

        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                input_ids = inputs,
                max_new_tokens = 300, # Espacio suficiente para el JSON y la explicación
                do_sample = False,
                pad_token_id = tokenizer.eos_token_id
            )

        res_bruta = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True).strip()

        print(f"EJEMPLO #{i+1} | ID: {m['id']} | Nivel: {m['nivel']}")
        print(f"SALIDA BRUTA DEL MODELO:\n{res_bruta}\n")

        # Intento de parseo para validar el formato
        try:
            # Limpiar posibles marcas de markdown ```json
            json_limpio = re.sub(r'```json\s*|\s*```', '', res_bruta).strip()
            res_json = json.loads(json_limpio)

            pred = res_json.get("respuesta", "").upper()
            expl = res_json.get("explicacion", "")

            print(f"📌 Letra extraída: {pred}")
            print(f"🎯 Respuesta correcta: {m['correcta']}")
            print(f"📝 Explicación: {expl}")

            if pred == m['correcta']:
                print("✅ RESULTADO: CORRECTO")
            else:
                print("❌ RESULTADO: INCORRECTO")

        except Exception as e:
            print(f"⚠️ ERROR AL PARSEAR EL JSON: {e}")

        print("-" * 60)

# Ejecutar la prueba
probar_formato_json(n_pruebas=2)

--- TEST DE FORMATO JSON Y EXPLICACIÓN (2 ejemplos) ---

EJEMPLO #1 | ID: A2_2016-02-19_E3_Q13 | Nivel: A2
SALIDA BRUTA DEL MODELO:
{
  "explicacion": "La opción C es correcta porque el texto menciona que se puede solicitar la revista en la web de la revista Casacien.",
  "respuesta": "C"
}

📌 Letra extraída: C
🎯 Respuesta correcta: C
📝 Explicación: La opción C es correcta porque el texto menciona que se puede solicitar la revista en la web de la revista Casacien.
✅ RESULTADO: CORRECTO
------------------------------------------------------------
EJEMPLO #2 | ID: B2_2005-11-19_E1_Q12 | Nivel: B2
SALIDA BRUTA DEL MODELO:
{
  "explicacion": "En el texto, Fernanda menciona que Enrique le ha escrito una carta y que su última carta no fue muy linda ni cariñosa, lo cual sugiere que Enrique está echando de menos algo o alguien, probablemente a Fernanda y a sus hijos.",
  "respuesta": "B"
}

📌 Letra extraída: B
🎯 Respuesta correcta: B
📝 Explicación: En el texto, Fernanda menciona que Enrique le

## [Mistral 7B Instruct](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)

In [4]:
max_seq_length = 4096
dtype = None
load_in_4bit = True

model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096, padding_idx=770)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_lay

### zero-shot

In [5]:
import json
import torch
import re
from tqdm import tqdm

# 1. Cargar ambos ficheros
with open('multiple_choice.json', 'r', encoding='utf-8') as f:
    main_data = json.load(f)

with open('subset_100.json', 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

def evaluar_modelo_subset():
    aciertos = 0
    total_procesadas = 0
    stats_por_nivel = {} # Para ver dónde falla más el modelo

    print(f"--- Iniciando evaluación sobre {len(ground_truth)} preguntas ---")

    for exam in main_data['exams']:
        nivel = exam['level']
        if nivel not in stats_por_nivel:
            stats_por_nivel[nivel] = {"aciertos": 0, "total": 0}

        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']
            texto_contexto = exercise.get('text', '')

            for question in exercise['questions']:
                q_id = question['questionId']

                # Solo procesamos si la pregunta está en nuestro subset de 100
                if q_id in ground_truth:
                    respuesta_correcta = ground_truth[q_id]

                    # Preparar opciones y Prompt
                    opciones_str = "\n".join([f"{o['optionId']}) {o['text']}" for o in question['options']])

                    messages = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": f"Texto:\n{texto_contexto}\n\nPregunta: {question['text']}\nOpciones:\n{opciones_str}\n\nRespuesta:"}
                    ]

                    # Inferencia
                    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

                    with torch.no_grad():
                        outputs = model.generate(
                            input_ids = inputs,
                            max_new_tokens = 5,
                            do_sample = False,
                            pad_token_id = tokenizer.eos_token_id
                        )

                    # Decodificar y limpiar
                    gen_tokens = outputs[0][len(inputs[0]):]
                    res_bruta = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip().upper()
                    match = re.search(r'[A-D]', res_bruta)
                    prediccion = match.group(0) if match else "N/A"

                    # Comparar
                    es_correcto = (prediccion == respuesta_correcta)
                    if es_correcto:
                        aciertos += 1
                        stats_por_nivel[nivel]["aciertos"] += 1

                    stats_por_nivel[nivel]["total"] += 1
                    total_procesadas += 1

    # --- RESULTADOS FINALES ---
    accuracy_total = (aciertos / total_procesadas) * 100 if total_procesadas > 0 else 0

    print("\n" + "="*40)
    print(f"📊 RESULTADOS GLOBALES")
    print(f"Total preguntas: {total_procesadas}")
    print(f"Aciertos: {aciertos}")
    print(f"Accuracy Total: {accuracy_total:.2f}%")
    print("="*40)

    print("\n📈 DESGLOSE POR NIVEL:")
    for nivel, stats in sorted(stats_por_nivel.items()):
        if stats['total'] > 0:
            acc_nivel = (stats['aciertos'] / stats['total']) * 100
            print(f"Nivel {nivel}: {acc_nivel:.2f}% ({stats['aciertos']}/{stats['total']})")

# Ejecutar la evaluación
evaluar_modelo_subset()

--- Iniciando evaluación sobre 100 preguntas ---


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 


📊 RESULTADOS GLOBALES
Total preguntas: 100
Aciertos: 73
Accuracy Total: 73.00%

📈 DESGLOSE POR NIVEL:
Nivel A1: 77.78% (28/36)
Nivel A2: 65.22% (15/23)
Nivel B1: 76.19% (16/21)
Nivel B2: 70.00% (14/20)


### zero-shot explicación

In [6]:
import json
import torch
import re
from tqdm import tqdm

path="/content/drive/MyDrive/resultados_mistral.json"

# --- 1. CONFIGURACIÓN DEL TOKENIZER PARA PROCESAMIENTO POR LOTES ---
# El padding a la izquierda es obligatorio para la generación de texto en lotes
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def ejecutar_batch_con_guardado(batch_size=4, output_file='resultados_detallados.json'):
    # Carga de datos
    with open('multiple_choice.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open('subset_100.json', 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)

    # Preparar la lista de tareas (solo las 100 del subset)
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']
            for q in exercise['questions']:
                q_id = q['questionId']
                if q_id in ground_truth:
                    opciones = "\n".join([f"{o['optionId']}) {o['text']}" for o in q['options']])
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": opciones,
                        "real": ground_truth[q_id]
                    })

    resultados_finales = []
    stats = {"total": 0, "aciertos": 0, "por_nivel": {}}

    print(f"🚀 Procesando {len(tareas)} preguntas en lotes de {batch_size}...")

    # Bucle de procesamiento por lotes
    for i in tqdm(range(0, len(tareas), batch_size), desc="Progreso"):
        batch = tareas[i : i + batch_size]

        # Construir mensajes para el lote
        batch_messages = [
            [
                {"role": "system", "content": SYSTEM_PROMPT_2},
                {"role": "user", "content": f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n{t['opciones']}"}
            ] for t in batch
        ]

        # 1. Tokenización y padding (Aseguramos que devuelva tensores y atención)
        model_inputs = tokenizer.apply_chat_template(
            batch_messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            padding=True,
            return_dict=True, # Importante para evitar el error de 'shape'
        ).to("cuda")

        # 2. Inferencia (Pasamos el diccionario desempaquetado)
        with torch.no_grad():
            outputs = model.generate(
                **model_inputs, # Usamos ** para pasar input_ids y attention_mask
                max_new_tokens=400,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # 3. Procesar resultados del lote
        # Usamos model_inputs.input_ids.shape[1] para saber dónde termina el prompt
        input_len = model_inputs.input_ids.shape[1]
        for j, output in enumerate(outputs):
            gen_tokens = output[input_len:]
            res_bruta = tokenizer.decode(gen_tokens, skip_special_tokens=True)

            # Extraer JSON y limpiar respuesta
            prediccion, explicacion = "ERROR", res_bruta
            try:
                json_match = re.search(r'\{.*\}', res_bruta, re.DOTALL)
                if json_match:
                    datos = json.loads(json_match.group(0))
                    letra_raw = datos.get("respuesta", "").strip().upper()
                    match_letra = re.search(r'[A-D]', letra_raw)
                    prediccion = match_letra.group(0) if match_letra else "N/A"
                    explicacion = datos.get("explicacion", "")
            except:
                pass

            # Evaluación y Estadísticas
            real = batch[j]["real"]
            nivel = batch[j]["nivel"]
            es_correcto = (prediccion == real)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1

            # Guardar en la lista para el fichero
            resultados_finales.append({
                "questionId": batch[j]["id"],
                "nivel": nivel,
                "pregunta": batch[j]["pregunta"],
                "respuesta_real": real,
                "prediccion_modelo": prediccion,
                "explicacion": explicacion,
                "estado": "CORRECTO" if es_correcto else "INCORRECTO"
            })

    # --- GUARDADO EN FICHERO ---
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(resultados_finales, f, ensure_ascii=False, indent=4)

    # --- INFORME DE ACCURACY ---
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*45)
    print(f"📊 RESULTADOS GUARDADOS EN: {output_file}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 45)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")
    print("="*45)

# Ejecutar el análisis
ejecutar_batch_con_guardado(batch_size=8, output_file=path)

🚀 Procesando 100 preguntas en lotes de 8...


Progreso:   0%|          | 0/13 [00:00<?, ?it/s]--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packa


📊 RESULTADOS GUARDADOS EN: /content/drive/MyDrive/resultados_mistral.json
Accuracy Global: 72.00% (72/100)
---------------------------------------------
Nivel A1: 80.56% (29/36)
Nivel A2: 60.87% (14/23)
Nivel B1: 66.67% (14/21)
Nivel B2: 75.00% (15/20)
